## Cleaning steps

1. Remove duplicates
2. Remove all row nulls
3. Fill the nulls/''/' '  
4. Trim spaces
5. Standardize case
6. Cast data types
7. Filter invalid rows
7. Rename columns
9. Add metadata columns if applicable
10. Validate schema

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

## Load table

In [0]:
df = spark.table('bikes_catalog.bronze.bronze_px_cat_g1v2')
df.show()

## Cleaning

In [0]:
df = df.dropDuplicates()

In [0]:
df = df.dropna(how='all')

In [0]:
for col in df.columns:
    print(f'Nulls in {col}: {df.filter(F.col(col).isNull()).count()}')

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name,F.trim(F.col(field.name)))

df.show()

In [0]:
for col in df.columns[1:]:
    print(f'Distinct values of {col}: {df.select(F.col(col)).distinct().count()}')
    print(f'{df.select(F.col(col)).distinct().show()}')

In [0]:
df.filter(df.SUBCAT == '').count()

In [0]:
df = df.withColumn(
    'MAINTENANCE',
    F.when(df.MAINTENANCE == 'Yes','Y')
    .when(df.MAINTENANCE == 'No','N')
    .otherwise(df.MAINTENANCE)
)
df.show()

In [0]:
for old_names, new_names in zip(df.columns,['Category_ID','Category','Sub_category','Maintainance']):
    df = df.withColumnRenamed(old_names,new_names)
df.show()

In [0]:
df.printSchema()

In [0]:
df.write.mode('overwrite').saveAsTable('bikes_catalog.silver.silver_px_cat_g1v2')